In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PROJECT_ROOT = Path(r"C:\Users\asus\OneDrive\EV-projects\evcs-projects")
BENCH_DIR    = PROJECT_ROOT / "results" / "benchmarking"
EXCEL_FILE   = BENCH_DIR / "benchmark_with_SLURM.xlsx"

df = pd.read_excel(EXCEL_FILE, sheet_name="benchmark")
print(f"Loaded {len(df)} rows")
df[['Timestamp','Instance','N','T','D_km','seed','DR_best','Exact_incumbent_raw','Gap_%']]

Loaded 13 rows


,Timestamp,Instance,N,T,D_km,seed,DR_best,Exact_incumbent_raw,Gap_%
0,2026-04-04 16:19:08,center_146_Verona_k250,250,6,2,11,1400.3634,1411.735909,0.8056
1,2026-04-05 13:25:38,center_79_Monza_k400,400,6,2,11,1978.9293,2009.965206,1.5441
2,2026-04-05 18:09:58,center_79_Monza_k400,400,6,2,11,1983.7417,2009.965206,1.3047
3,2026-04-05 18:46:48,center_247_Reggio_Emilia_k125,125,6,2,11,749.4384,749.960492,0.0696
4,2026-04-05 18:46:49,center_102_Vicenza_k125,125,6,2,11,751.2292,751.733425,0.0671
5,2026-04-05 18:48:13,center_153_Padova_k175,175,6,2,11,1034.3764,1035.779556,0.1355
6,2026-04-05 18:48:39,center_240_Parma_k200,200,6,2,11,1160.7179,1161.544692,0.0712
7,2026-04-05 18:49:08,center_323_Prato_k200,200,6,2,11,1160.5832,1161.346191,0.0657
8,2026-04-05 18:49:52,center_58_Trieste_k225,225,6,2,11,1284.3931,1285.238826,0.0658
9,2026-04-05 18:49:55,center_146_Verona_k250,250,6,2,11,1408.6489,1411.735909,0.2187


In [ ]:
xl = pd.ExcelFile(EXCEL_FILE)
TRACE_SHEETS = sorted(
    [s for s in xl.sheet_names if s.startswith("t") and s[1:].isdigit()],
    key=lambda s: int(s[1:])
)
print("Trace sheets:", TRACE_SHEETS)


def get_trace_for_sheet(sheet_name):
    return xl.parse(sheet_name)


def row_for_sheet(sheet_name):
    """Return the benchmark row for this trace sheet, or None if not found."""
    idx = int(sheet_name[1:])
    return df.loc[idx] if idx in df.index else None


def plot_dr_curve(ax, trace, row, sheet_name):
    x    = trace["iteration"].to_numpy()
    best = trace["best_full"].ffill().to_numpy()

    # Fluctuating current line
    if "proxy_mean" in trace.columns:
        ax.plot(x, trace["proxy_mean"].to_numpy(), linewidth=1.0, alpha=0.55,
                color="tab:blue", label="proxy mean")
    elif "proxy_max" in trace.columns:
        ax.plot(x, trace["proxy_max"].to_numpy(), linewidth=1.0, alpha=0.55,
                color="tab:blue", label="proxy max")

    # DR best-so-far
    ax.plot(x, best, linewidth=2.5, color="tab:orange",
            label=f"DR best ({best[-1]:.3f})")

    if row is not None:
        instance = row["Instance"]
        N        = int(row["N"])
        exact    = row["Exact_incumbent_raw"]
        gap      = row["Gap_%"]

        if pd.notna(exact):
            ax.axhline(y=float(exact), linestyle="--", linewidth=2.0,
                       color="tab:red", label=f"Exact ({float(exact):.3f})")

        gap_str = f"{gap:.4f}%" if pd.notna(gap) else "N/A"
        title = (f"{instance}  |  N={N}\n"
                 f"Gap={gap_str}   DR={best[-1]:.3f}   iters={len(x)}")
    else:
        title = f"Sheet {sheet_name}  (no benchmark metadata)\niters={len(x)}"

    ax.set_title(title, fontsize=9)
    ax.set_xlabel("iteration", fontsize=9)
    ax.set_ylabel("Score", fontsize=9)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    ax.grid(True, linewidth=0.5)


print("Helper functions ready.")


In [ ]:
plottable = []
for sh in TRACE_SHEETS:
    trace = get_trace_for_sheet(sh)
    row   = row_for_sheet(sh)
    if row is not None:
        label = f"{row['Instance']} (N={int(row['N'])})"
    else:
        label = f"sheet {sh} — no benchmark metadata"
    print(f"  {sh}: {label}")
    plottable.append((sh, row, trace))

print(f"\n{len(plottable)} trace sheet(s) to plot.")


In [ ]:
ncols = 2
nrows = int(np.ceil(len(plottable) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 5 * nrows))
axes = np.array(axes).reshape(-1)

for i, (sh, row, trace) in enumerate(plottable):
    plot_dr_curve(axes[i], trace, row, sh)

for i in range(len(plottable), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()


In [6]:
# --- summary table with color-coded gap ---
cols = ['Timestamp','Instance','Policy','N','T','D_km','seed',
        'Exact_incumbent_raw','DR_best','Gap_%','DR_iters','DR_time_s','Exact_time_s']
show = [c for c in cols if c in df.columns]
df[show].style \
    .format({'Exact_incumbent_raw':'{:.4f}','DR_best':'{:.4f}',
             'Gap_%':'{:.4f}%','DR_time_s':'{:.1f}s','Exact_time_s':'{:.1f}s'}) \
    .background_gradient(subset=['Gap_%'], cmap='RdYlGn_r')

,Timestamp,Instance,Policy,N,T,D_km,seed,Exact_incumbent_raw,DR_best,Gap_%,DR_iters,DR_time_s,Exact_time_s
0,2026-04-04 16:19:08,center_146_Verona_k250,closest_priority,250,6,2,11,1411.7359,1400.3634,0.8056%,10,332.2s,187.2s
1,2026-04-05 13:25:38,center_79_Monza_k400,closest_priority,400,6,2,11,2009.9652,1978.9293,1.5441%,7,337.7s,954.1s
2,2026-04-05 18:09:58,center_79_Monza_k400,closest_priority,400,6,2,11,2009.9652,1983.7417,1.3047%,22,1030.8s,1023.0s
3,2026-04-05 18:46:48,center_247_Reggio_Emilia_k125,closest_priority,125,6,2,11,749.9605,749.4384,0.0696%,51,1003.8s,17.4s
4,2026-04-05 18:46:49,center_102_Vicenza_k125,closest_priority,125,6,2,11,751.7334,751.2292,0.0671%,70,1010.1s,12.3s
5,2026-04-05 18:48:13,center_153_Padova_k175,closest_priority,175,6,2,11,1035.7796,1034.3764,0.1355%,44,1027.0s,79.0s
6,2026-04-05 18:48:39,center_240_Parma_k200,closest_priority,200,6,2,11,1161.5447,1160.7179,0.0712%,28,1033.1s,99.5s
7,2026-04-05 18:49:08,center_323_Prato_k200,closest_priority,200,6,2,11,1161.3462,1160.5832,0.0657%,29,1021.1s,140.5s
8,2026-04-05 18:49:52,center_58_Trieste_k225,closest_priority,225,6,2,11,1285.2388,1284.3931,0.0658%,24,1012.9s,193.4s
9,2026-04-05 18:49:55,center_146_Verona_k250,closest_priority,250,6,2,11,1411.7359,1408.6489,0.2187%,31,1015.8s,193.4s
